# API SaaS Financial Model — Executive Dashboard

This notebook reads the pre-computed metrics from `data/outputs/metrics.csv`
(generated by `scripts/compute_metrics.py`) and produces visualisations
suitable for an executive review or board deck.

**Run the pipeline first:**
```bash
python scripts/generate_inputs.py
python scripts/compute_metrics.py
```

In [ ]:
from pathlib import Path
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker

ROOT = Path(".").resolve().parent
METRICS_PATH = ROOT / "data" / "outputs" / "metrics.csv"
INPUTS_PATH  = ROOT / "data" / "outputs" / "simulated_inputs.csv"

df = pd.read_csv(METRICS_PATH)
df_inputs = pd.read_csv(INPUTS_PATH)

print(f"Loaded {len(df)} months of metrics and {len(df_inputs)} tier-month rows of inputs.")
df.head(3)

## 1. Revenue Growth — MRR & ARR

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 4))

axes[0].bar(df["month"], df["mrr"] / 1_000, color="#1f4e79", alpha=0.85)
axes[0].set_title("Monthly Recurring Revenue (MRR)", fontsize=13, fontweight="bold")
axes[0].set_xlabel("Month")
axes[0].set_ylabel("MRR ($k)")
axes[0].tick_params(axis="x", rotation=45)
axes[0].yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f"${x:,.0f}k"))
axes[0].set_xticks(df["month"][::6])

axes[1].plot(df["month"], df["arr"] / 1_000_000, marker="o", color="#2e75b6", linewidth=2)
axes[1].set_title("Annual Recurring Revenue (ARR)", fontsize=13, fontweight="bold")
axes[1].set_xlabel("Month")
axes[1].set_ylabel("ARR ($M)")
axes[1].tick_params(axis="x", rotation=45)
axes[1].yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f"${x:,.1f}M"))
axes[1].set_xticks(df["month"][::6])

plt.tight_layout()
plt.show()

## 2. Revenue Mix — Subscription vs. Overage

In [ ]:
fig, ax = plt.subplots(figsize=(12, 4))

ax.stackplot(
    df["month"],
    df["subscription_revenue"] / 1_000,
    df["overage_revenue"] / 1_000,
    labels=["Subscription", "Overage"],
    colors=["#1f4e79", "#70ad47"],
    alpha=0.85,
)
ax.set_title("Revenue Mix: Subscription vs. Overage", fontsize=13, fontweight="bold")
ax.set_xlabel("Month")
ax.set_ylabel("Revenue ($k)")
ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f"${x:,.0f}k"))
ax.set_xticks(df["month"][::6])
ax.tick_params(axis="x", rotation=45)
ax.legend(loc="upper left")
plt.tight_layout()
plt.show()

## 3. Gross Margin

In [ ]:
fig, ax = plt.subplots(figsize=(12, 4))

ax.plot(df["month"], df["gross_margin_pct"], color="#70ad47", linewidth=2, marker="o", markersize=4)
ax.axhline(60, color="red", linestyle="--", linewidth=1, label="Floor (60%)")
ax.fill_between(df["month"], df["gross_margin_pct"], 60,
                where=(df["gross_margin_pct"] >= 60),
                alpha=0.15, color="#70ad47", label="Above floor")
ax.set_title("Blended Gross Margin %", fontsize=13, fontweight="bold")
ax.set_xlabel("Month")
ax.set_ylabel("Gross Margin (%)")
ax.set_ylim(0, 100)
ax.set_xticks(df["month"][::6])
ax.tick_params(axis="x", rotation=45)
ax.legend()
plt.tight_layout()
plt.show()

## 4. P&L Waterfall — Revenue, Gross Profit, EBITDA

In [ ]:
fig, ax = plt.subplots(figsize=(12, 4))

ax.plot(df["month"], df["total_revenue"] / 1_000, label="Revenue",      color="#1f4e79", linewidth=2)
ax.plot(df["month"], df["gross_profit"] / 1_000, label="Gross Profit",  color="#2e75b6", linewidth=2)
ax.plot(df["month"], df["ebitda"]       / 1_000, label="EBITDA",        color="#70ad47", linewidth=2)
ax.axhline(0, color="black", linewidth=0.8, linestyle="--")

ax.set_title("P&L Summary", fontsize=13, fontweight="bold")
ax.set_xlabel("Month")
ax.set_ylabel("Amount ($k)")
ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f"${x:,.0f}k"))
ax.set_xticks(df["month"][::6])
ax.tick_params(axis="x", rotation=45)
ax.legend()
plt.tight_layout()
plt.show()

## 5. Burn Rate & Cumulative Cash

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 4))

axes[0].bar(df["month"], df["burn_rate"] / 1_000, color="#c00000", alpha=0.85)
axes[0].set_title("Monthly Burn Rate", fontsize=13, fontweight="bold")
axes[0].set_xlabel("Month")
axes[0].set_ylabel("Burn ($k)")
axes[0].yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f"${x:,.0f}k"))
axes[0].set_xticks(df["month"][::6])
axes[0].tick_params(axis="x", rotation=45)

colors = ["#70ad47" if v >= 0 else "#c00000" for v in df["cumulative_net_cash"]]
axes[1].bar(df["month"], df["cumulative_net_cash"] / 1_000, color=colors, alpha=0.85)
axes[1].axhline(0, color="black", linewidth=0.8)
axes[1].set_title("Cumulative Net Cash Position", fontsize=13, fontweight="bold")
axes[1].set_xlabel("Month")
axes[1].set_ylabel("Cumulative Cash ($k)")
axes[1].yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f"${x:,.0f}k"))
axes[1].set_xticks(df["month"][::6])
axes[1].tick_params(axis="x", rotation=45)

plt.tight_layout()
plt.show()

## 6. Paying Customer Growth by Tier

In [ ]:
tier_colors = {"Free": "#bdd7ee", "Starter": "#2e75b6", "Growth": "#1f4e79", "Enterprise": "#70ad47"}

fig, ax = plt.subplots(figsize=(12, 4))
for tier in ["Starter", "Growth", "Enterprise"]:
    tier_df = df_inputs[df_inputs["tier"] == tier]
    ax.plot(
        tier_df["month"],
        tier_df["customers_eom"],
        label=tier,
        color=tier_colors[tier],
        linewidth=2,
        marker="o",
        markersize=3,
    )

ax.set_title("Paying Customer Growth by Tier", fontsize=13, fontweight="bold")
ax.set_xlabel("Month")
ax.set_ylabel("Customers (EOM)")
ax.set_xticks(tier_df["month"][::6])
ax.tick_params(axis="x", rotation=45)
ax.legend()
plt.tight_layout()
plt.show()

## 7. Unit Economics Summary

In [ ]:
paid_tiers = ["Starter", "Growth", "Enterprise"]
last = df.iloc[-1]

ue_data = []
for tier in paid_tiers:
    prefix = tier.lower()
    ue_data.append({
        "Tier": tier,
        "LTV ($)": f"${last[f'{prefix}_ltv']:,.0f}",
        "CAC ($)": f"${last[f'{prefix}_cac']:,.0f}",
        "LTV:CAC": f"{last[f'{prefix}_ltv_cac']:.1f}×",
        "Payback (months)": f"{last[f'{prefix}_payback_months']:.1f}",
    })

ue_df = pd.DataFrame(ue_data).set_index("Tier")
print(ue_df.to_string())

## 8. KPI Summary Table

In [ ]:
snapshots = []
for month_idx in [12, 24, 36]:
    row = df[df["month_index"] == month_idx]
    if row.empty:
        continue
    r = row.iloc[0]
    snapshots.append({
        "Month": int(r["month_index"]),
        "Period": r["month"],
        "MRR": f"${r['mrr']:,.0f}",
        "ARR": f"${r['arr']:,.0f}",
        "Gross Margin": f"{r['gross_margin_pct']:.1f}%",
        "EBITDA": f"${r['ebitda']:,.0f}",
        "Paying Customers": int(r["paying_customers"]),
        "Cumulative Cash": f"${r['cumulative_net_cash']:,.0f}",
    })

kpi_df = pd.DataFrame(snapshots).set_index("Month")
print(kpi_df.to_string())